In [ ]:
!pip install transformers datasets torch accelerate gradio huggingface_hub -q

print("All libraries installed successfully!")

In [ ]:
text = """
Artificial Intelligence is transforming industries.

Machine learning helps computers learn patterns.

Python is widely used for AI development.

Deep learning models require large datasets.

Technology is improving healthcare and education.

Learning to code improves problem-solving skills.

Natural Language Processing helps computers understand text.

Cybersecurity is important in the digital world.

Cloud computing enables scalable applications.

Data science combines statistics and programming.
"""

In [ ]:
with open("dataset.txt", "w") as f:
    f.write(text)

print("Dataset file created successfully!")

In [ ]:
with open("dataset.txt", "r") as f:
    print(f.read())

In [ ]:
!ls

In [ ]:
!pip install transformers datasets torch accelerate gradio huggingface_hub -q

print("Libraries installed successfully!")

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = "gpt2"

# Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Model
model = GPT2LMHeadModel.from_pretrained(model_name)

print("GPT-2 loaded successfully!")

In [ ]:
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling

# Load dataset
dataset = load_dataset("text", data_files={"train": "dataset.txt"})

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Tokenize dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Dataset prepared successfully!")

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

print("Training configuration ready!")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"]
)

trainer.train()

In [ ]:
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

print("Model saved successfully!")

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load saved model
model = GPT2LMHeadModel.from_pretrained("./gpt2-finetuned")

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-finetuned")

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

print("Fine-tuned model loaded!")

In [ ]:
def generate_text(prompt):

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True,
        top_k=40,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return generated_text

In [ ]:
print(generate_text("Artificial Intelligence"))

print()

print(generate_text("Python programming"))

print()

print(generate_text("Machine learning"))

In [ ]:
import gradio as gr

# Chatbot function
def chatbot(prompt):
    return generate_text(prompt)

# Gradio interface
iface = gr.Interface(
    fn=chatbot,

    inputs=gr.Textbox(
        lines=2,
        placeholder="Enter your prompt here...",
        label="Prompt"
    ),

    outputs=gr.Textbox(
        label="Generated Text"
    ),

    title="🚀 Custom GPT-2 AI Chatbot",

    description="Fine-tuned GPT-2 model using custom dataset"
)

iface.launch(share=True)

In [ ]:
iface.launch(share=True)

In [ ]:
!zip -r gpt2-finetuned.zip gpt2-finetuned

In [ ]:
from google.colab import files

files.download("gpt2-finetuned.zip")